# CRNN Baseline — Kaggle GPU Training

Trains the Devanagari CRNN baseline on a Kaggle T4.

Dataset: `sanskritipoudel/devnagari-preprocessed` (attached). Code: repo branch `ml`.
Run all cells top to bottom. Outputs land in `/kaggle/working`.

In [ ]:
# 1. Confirm GPU is on
import torch
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code (branch: ml). cv2/numpy/tqdm are preinstalled on Kaggle.
import os
if not os.path.isdir('/kaggle/working/Devnagari_Handwriting_Recognition'):
    !git clone -b ml https://github.com/Sanskriti-poudel/Devnagari_Handwriting_Recognition.git /kaggle/working/Devnagari_Handwriting_Recognition
%cd /kaggle/working/Devnagari_Handwriting_Recognition
!git log --oneline -1

In [ ]:
# 3. Locate the dataset root (the folder that directly contains train/) and point the loader at it.
import os

def find_data_root(base='/kaggle/input'):
    for dirpath, dirnames, _ in os.walk(base):
        if 'train' in dirnames and os.path.isdir(os.path.join(dirpath, 'train')):
            return dirpath
    raise FileNotFoundError(f'No folder containing train/ found under {base}')

DATA_ROOT = find_data_root()
os.environ['DEVNAGARI_DATA_ROOT'] = DATA_ROOT
print('DEVNAGARI_DATA_ROOT =', DATA_ROOT)
print('contents:', os.listdir(DATA_ROOT))
# Sanity: resolve the first training path from the split index
import json
split = json.load(open('data/split_index.json'))
sample = os.path.join(DATA_ROOT, split['train'][0])
print('sample path:', sample, '-> exists:', os.path.exists(sample))

In [ ]:
# 4. Train (auto-detects CUDA; batch_size defaults to 64 on GPU).
#    Override with CRNN_BATCH_SIZE if you hit out-of-memory.
!python models/crnn/train.py

In [ ]:
# 5. Copy outputs to /kaggle/working root so they're easy to download from the version.
import shutil, os
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
for src in ['models/crnn/checkpoints/best_model.pth', 'logs/crnn_training.csv']:
    if os.path.exists(src):
        shutil.copy(src, '/kaggle/working/artifacts/')
        print('saved', src)
    else:
        print('MISSING', src)
print(os.listdir('/kaggle/working/artifacts'))